In [1]:
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple
import os
import json
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

/Users/egor/VS_GIT_repositories/BYTE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = '/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet'
CHROMA_DB_PATH = '/Users/egor/VS_GIT_repositories/BYTE/src/RAG/chroma_db'
EMBEDDING_MODEL = 'cointegrated/rubert-tiny2'
BATCH_SIZE = 32
TOP_K = 5 

Path(CHROMA_DB_PATH).mkdir(parents=True, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"ChromaDB path: {CHROMA_DB_PATH}")
print(f"Embedding model: {EMBEDDING_MODEL}")

Data path: /Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet
ChromaDB path: /Users/egor/VS_GIT_repositories/BYTE/src/RAG/chroma_db
Embedding model: cointegrated/rubert-tiny2


In [3]:
df = pd.read_parquet(DATA_PATH)
df.head()

,offer_id,price,price_numeric,old_price,area,rooms,floor,price_per_m2,metro,metro_time,...,main_image,photo_count,badges,publish_date,url,title,description,image_urls,self_floor,max_floor
0,7035113340557126091,7 500 000 ₽,7500000.0,NaN,17.7,студия,9 этаж из 16,None,Калитники,9.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,None,https://realty.yandex.ru/offer/703511334055712...,апартаменты-студия,Номер лота: 99696. Панорамный вид из больших о...,https://avatars.mds.yandex.net/get-realty-offe...,16.0,16.0
1,7035113340416809607,7 500 000 ₽,7500000.0,NaN,17.0,студия,2 этаж из 2,None,Соколиная гора,8.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/703511334041680...,апартаменты-студия,Номер лота: 87440. Продается студия с дизайнер...,https://avatars.mds.yandex.net/get-realty-offe...,2.0,2.0
2,7053956964805047621,12 200 000 ₽,12200000.0,NaN,17.9,студия,2 этаж из 48,None,Тушинская,10.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,3 квартал 2027,https://realty.yandex.ru/offer/705395696480504...,квартира-студия,"Арт. 119802099 Студия 17,9 м в CITYZEN Урбан-б...",https://avatars.mds.yandex.net/get-realty-offe...,48.0,48.0
3,7053956914445503237,7 300 000 ₽,7300000.0,NaN,15.7,студия,5 этаж из 5,None,Бутырская,17.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/705395691444550...,квартира-студия,Арт. 134010491 СПЕЦИАЛЬНО для наших клиентов с...,https://avatars.mds.yandex.net/get-realty-offe...,5.0,5.0
4,3699730400767130013,10 802 031 ₽,10802031.0,NaN,14.1,студия,5 этаж из 16,None,Коммунарка,14.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,2 квартал 2026,https://realty.yandex.ru/offer/369973040076713...,квартира-студия,Строим кварталы для жизни с заботой о будущем....,https://avatars.mds.yandex.net/get-realty-offe...,16.0,16.0


In [4]:
df['text_for_embedding'] = (
    df['title'].fillna('').astype(str) + ' ' +
    # df['description'].fillna('').astype(str) + ' ' +
    df['address'].fillna('').astype(str) + ' ' +
    'Метро: ' + df['metro'].fillna('').astype(str) + ' ' +
    'Время до метро: ' + df['metro_time'].fillna('').astype(str) + ' ' +
    'Комнаты: ' + df['rooms'].astype(str) + ' ' +
    'Площадь: ' + df['area'].astype(str) + ' м2 ' +
    'Цена: ' + df['price'].astype(str) + ' руб ' +
    'Этаж: ' + df['floor'].astype(str)
).str.strip()
print(df['text_for_embedding'].iloc[0])

апартаменты-студия Москва, Подъёмная улица, ,  , 1 Метро: Калитники Время до метро: 9.0 Комнаты: студия Площадь: 17.7 м2 Цена: 7 500 000 ₽ руб Этаж: 9 этаж из 16


In [5]:
print("Loading model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Emb dimension: {embedding_model.get_embedding_dimension()}")

Loading model...


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 13736.26it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Emb dimension: 312


In [6]:
texts = df['text_for_embedding'].tolist()
embeddings = embedding_model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f" Embeddings generated. Shape: {embeddings.shape}")

Batches: 100%|██████████| 2928/2928 [01:45<00:00, 27.75it/s]


 Embeddings generated. Shape: (93667, 312)


In [7]:
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

collection_name = "nedvijimost"
try:
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
except Exception:
    pass

Deleted existing collection: nedvijimost


In [8]:
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

print(f" Collection created: {collection_name}")

 Collection created: nedvijimost


In [9]:
metadata_list = []
documents_list = []
ids_list = []
embeddings_list = []

for idx, row in df.iterrows():
    metadata_list.append({
        "offer_id": str(row['offer_id']),
        "price": float(row['price_numeric']) if pd.notna(row['price_numeric']) else 0,
        "area": float(row['area']) if pd.notna(row['area']) else 0,
        "rooms": str(row['rooms']),
        "metro": str(row['metro']),
        "address": str(row['address']),
        "url": str(row['url']),
        "title": str(row['title'])[:200]
    })
    documents_list.append(row['text_for_embedding'])
    ids_list.append(f"apt_{row['offer_id']}")
    embeddings_list.append(embeddings[idx])

In [10]:
batch_size = 100
for i in tqdm(range(0, len(ids_list), batch_size), desc="Adding to ChromaDB"):
    batch_end = min(i + batch_size, len(ids_list))
    collection.add(
        ids=ids_list[i:batch_end],
        embeddings=embeddings_list[i:batch_end],
        metadatas=metadata_list[i:batch_end],
        documents=documents_list[i:batch_end]
    )

print(f"Added {len(ids_list)} objects to ChromaDB")

Adding to ChromaDB: 100%|██████████| 937/937 [02:23<00:00,  6.54it/s]

Added 93667 objects to ChromaDB


## Task 2: Semantic Search Function

Function to find similar apartments based on arbitrary text input.

In [11]:
def search_similar_apartments(
    query: str,
    top_k: int = TOP_K,
    embedding_model = embedding_model,
    collection = collection
):
    
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0].tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    similar_apartments = []
    for i in range(len(results['ids'][0])):
        similar_apartments.append({
            'id': results['ids'][0][i],
            'distance': results['distances'][0][i],
            'metadata': results['metadatas'][0][i],
            'document': results['documents'][0][i]
        })
    
    return similar_apartments

In [12]:
test_query = "Студия в центре с хорошей транспортной доступностью до 8 млн"
print(f"Test query: {test_query}\n")
results = search_similar_apartments(test_query, top_k=3, embedding_model=embedding_model, collection=collection)

for i, apt in enumerate(results, 1):
    print(f"{i}. Offer ID: {apt['metadata']['offer_id']}")
    print(f"   Similarity distance: {apt['distance']:.4f}")
    print(f"   Price: ₽ {apt['metadata']['price']:,.0f}")
    print(f"   Address: {apt['metadata']['address']}")
    print(f"   URL: {apt['metadata']['url']}\n")

Test query: Студия в центре с хорошей транспортной доступностью до 8 млн

1. Offer ID: 6638757692273330338
   Similarity distance: 0.3082
   Price: ₽ 8,800,000
   Address: Апарт-комплекс HighWay
   URL: https://realty.yandex.ru/offer/6638757692273330338/

2. Offer ID: 8681917785575143745
   Similarity distance: 0.3152
   Price: ₽ 12,000,000
   Address: Апарт-комплекс HighWay
   URL: https://realty.yandex.ru/offer/8681917785575143745/

3. Offer ID: 6299749192785639974
   Similarity distance: 0.3174
   Price: ₽ 6,800,000
   Address: Апарт-комплекс N’ICE LOFT
   URL: https://realty.yandex.ru/offer/6299749192785639974/



## Task 4: DVC Tracking

Version control vector database and embeddings artifacts using DVC.

In [13]:
import pickle
import json

rag_artifacts_dir = Path('/Users/egor/VS_GIT_repositories/BYTE/src/RAG/artifacts')
rag_artifacts_dir.mkdir(parents=True, exist_ok=True)

embeddings_path = rag_artifacts_dir / 'embeddings.npy'
np.save(embeddings_path, embeddings)
print(f" Saved embeddings to {embeddings_path}")

metadata_path = rag_artifacts_dir / 'metadata.json'
metadata_json = []
for meta in metadata_list:
    meta_copy = meta.copy()
    metadata_json.append(meta_copy)

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata_json, f, ensure_ascii=False, indent=2)
print(f"✅ Saved metadata to {metadata_path}")

config = {
    'embedding_model': EMBEDDING_MODEL,
    'total_apartments': len(df),
    'embedding_dimension': int(embedding_model.get_sentence_embedding_dimension()),
    'chroma_db_path': CHROMA_DB_PATH,
    'created_at': pd.Timestamp.now().isoformat()
}
config_path = rag_artifacts_dir / 'config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)
print(f" Saved config to {config_path}")

print(f"\n All artifacts saved to: {rag_artifacts_dir}")

 Saved embeddings to /Users/egor/VS_GIT_repositories/BYTE/src/RAG/artifacts/embeddings.npy
✅ Saved metadata to /Users/egor/VS_GIT_repositories/BYTE/src/RAG/artifacts/metadata.json
 Saved config to /Users/egor/VS_GIT_repositories/BYTE/src/RAG/artifacts/config.json

 All artifacts saved to: /Users/egor/VS_GIT_repositories/BYTE/src/RAG/artifacts


In [14]:
import subprocess

project_root = Path('/Users/egor/VS_GIT_repositories/BYTE')

print(" DVC Commands to run in terminal:")
print("\n" + "="*80)
print("# Add artifacts to DVC tracking")
print(f"cd {project_root}")
print(f"dvc add src/RAG/artifacts/embeddings.npy")
print(f"dvc add src/RAG/artifacts/metadata.json")
print(f"dvc add src/RAG/artifacts/config.json")
print(f"dvc add src/RAG/chroma_db")
print("\n# Commit changes to git")
print("git add src/RAG/artifacts/*.dvc src/RAG/chroma_db.dvc .gitignore")
print("git commit -m 'Add RAG artifacts: embeddings, metadata, vector database'")
print("="*80 + "\n")

print(" RAG Pipeline Ready!")
print("\n Summary:")
print(f"  • Embeddings: {embeddings.shape}")
print(f"  • Apartments indexed: {len(df)}")
print(f"  • Embedding model: {EMBEDDING_MODEL}")
print(f"  • Vector DB: ChromaDB at {CHROMA_DB_PATH}")
print(f"  • Artifacts: {rag_artifacts_dir}")

 DVC Commands to run in terminal:

# Add artifacts to DVC tracking
cd /Users/egor/VS_GIT_repositories/BYTE
dvc add src/RAG/artifacts/embeddings.npy
dvc add src/RAG/artifacts/metadata.json
dvc add src/RAG/artifacts/config.json
dvc add src/RAG/chroma_db

# Commit changes to git
git add src/RAG/artifacts/*.dvc src/RAG/chroma_db.dvc .gitignore
git commit -m 'Add RAG artifacts: embeddings, metadata, vector database'

 RAG Pipeline Ready!

 Summary:
  • Embeddings: (93667, 312)
  • Apartments indexed: 93667
  • Embedding model: cointegrated/rubert-tiny2
  • Vector DB: ChromaDB at /Users/egor/VS_GIT_repositories/BYTE/src/RAG/chroma_db
  • Artifacts: /Users/egor/VS_GIT_repositories/BYTE/src/RAG/artifacts
